# Neural Trees

## Data Mining Project

**Authors:** Aya Abdine and Meriam El Askri  
**Institution:** Constructor University  
**Course:** Data Mining

This notebook develops a Neural Trees project step by step. The goal is to load the datasets, explore them, prepare the features, train Neural Tree models, compare them with common machine-learning baselines, and summarize the results clearly.

# 1. Imports and Configuration

We start by importing the libraries used for data preparation, Neural Tree models, baseline models, metrics, and plots.

In [ ]:
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
import matplotlib

if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")

import matplotlib.pyplot as plt

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, r2_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

### Optional XGBoost

In [ ]:
try:
    from xgboost import XGBClassifier, XGBRegressor
    has_xgb = True
except ImportError:
    has_xgb = False

print("XGBoost available:", has_xgb)

### Reproducibility Settings

In [ ]:
SEED = 42
MAX_ROWS = 1000
EPOCHS = 40

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_PATH = Path("data")

print("Device:", device)
print("Data path:", DATA_PATH.resolve())

# 2. Neural Tree and Neural Boosting Models

The Neural Tree is a soft decision tree. Each internal node makes a differentiable split using a sigmoid, and each leaf stores a prediction. A tree of depth 2 has 3 split nodes and 4 leaves.

### Build the Paths from Root to Leaves

In [ ]:
def make_paths(depth):
    paths = []

    def visit(node, level, path):
        if level == depth:
            paths.append(path)
            return
        visit(2 * node + 1, level + 1, path + [(node, 0)])
        visit(2 * node + 2, level + 1, path + [(node, 1)])

    visit(0, 0, [])
    return paths

### Define the Neural Tree

In [ ]:
class NeuralTree(nn.Module):
    def __init__(self, input_size, output_size, depth=2):
        super().__init__()
        self.paths = make_paths(depth)
        self.splits = nn.Linear(input_size, 2**depth - 1)
        self.leaves = nn.Parameter(torch.randn(2**depth, output_size) * 0.01)

    def forward(self, x):
        split_p = torch.sigmoid(self.splits(x))
        leaf_p = []
        for path in self.paths:
            p = torch.ones(x.shape[0], device=x.device)
            for node, side in path:
                p = p * (split_p[:, node] if side else 1 - split_p[:, node])
            leaf_p.append(p)
        return torch.stack(leaf_p, dim=1) @ self.leaves

This is the main Neural Tree model used in the notebook. The code follows the model idea directly: soft split nodes send each sample to the leaves with learned probabilities.

# 3. Load Datasets

### Iris Classification

In [ ]:
iris_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]
iris = pd.read_csv(DATA_PATH / "iris" / "iris.data", names=iris_cols).dropna()

iris_X = iris.drop("species", axis=1)
iris_y = iris["species"]

### Wine Quality Regression

In [ ]:
red = pd.read_csv(DATA_PATH / "wine+quality" / "winequality-red.csv", sep=";")
white = pd.read_csv(DATA_PATH / "wine+quality" / "winequality-white.csv", sep=";")

red["type"] = "red"
white["type"] = "white"
wine = pd.concat([red, white], ignore_index=True)

In [ ]:
wine_X = wine.drop("quality", axis=1)
wine_y = wine["quality"]

### Student Performance Regression

In [ ]:
student = pd.read_csv(DATA_PATH / "student+performance" / "student-mat.csv", sep=";")

student_X = student.drop("G3", axis=1)
student_y = student["G3"]

### Adult Income Classification

In [ ]:
adult_cols = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country", "income",
]

In [ ]:
adult = pd.read_csv(DATA_PATH / "adult" / "adult.data", names=adult_cols,
                    na_values=" ?", skipinitialspace=True)
adult = adult.dropna()

adult_X = adult.drop("income", axis=1)
adult_y = adult["income"]

### Bank Marketing Classification

In [ ]:
bank = pd.read_csv(DATA_PATH / "bank+marketing" / "bank-full.csv", sep=";")

bank_X = bank.drop("y", axis=1)
bank_y = bank["y"]

### Breast Cancer Classification

In [ ]:
wdbc_cols = ["id", "diagnosis"] + [f"feature_{i}" for i in range(1, 31)]
wdbc = pd.read_csv(DATA_PATH / "breast+cancer+wisconsin+diagnostic" / "wdbc.data",
                   names=wdbc_cols)

cancer_X = wdbc.drop(["id", "diagnosis"], axis=1)
cancer_y = wdbc["diagnosis"]

### Dataset Summary

In [ ]:
datasets = [
    ("Iris", "classification", iris_X, iris_y),
    ("Wine Quality", "regression", wine_X, wine_y),
    ("Student Performance", "regression", student_X, student_y),
    ("Adult", "classification", adult_X, adult_y),
    ("Bank Marketing", "classification", bank_X, bank_y),
    ("Breast Cancer WDBC", "classification", cancer_X, cancer_y),
]

In [ ]:
summary_rows = []

for name, task, X_data, y_data in datasets:
    summary_rows.append([name, task, X_data.shape[0], X_data.shape[1], y_data.nunique()])

dataset_summary = pd.DataFrame(summary_rows, columns=["dataset", "task", "rows", "features", "targets"])
dataset_summary

The project uses both classification and regression datasets. Larger datasets are sampled during experiments so the full notebook remains practical to run.

# 4. First Dataset Walkthrough: Iris

### Explore Iris

In [ ]:
iris.head()

In [ ]:
iris.describe()

In [ ]:
iris["species"].value_counts()

Iris is balanced across the three species, so accuracy and weighted F1 are both meaningful.

### Prepare Iris Labels

In [ ]:
y_codes = iris_y.astype("category").cat.codes
iris_classes = list(iris_y.astype("category").cat.categories)

iris_classes

### Split and Scale Iris

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    iris_X, y_codes, test_size=0.2, random_state=SEED, stratify=y_codes
)

In [ ]:
scaler = StandardScaler()

X_train_p = scaler.fit_transform(X_train)
X_test_p = scaler.transform(X_test)

### Create PyTorch Data

In [ ]:
X_train_t = torch.tensor(X_train_p, dtype=torch.float32)
y_train_t = torch.tensor(y_train.to_numpy(), dtype=torch.long)

train_ds = TensorDataset(X_train_t, y_train_t)
loader = DataLoader(train_ds, batch_size=16, shuffle=True)

### Train a Single Neural Tree

In [ ]:
single_tree = NeuralTree(4, 3, depth=2).to(device)
opt = torch.optim.Adam(single_tree.parameters(), lr=0.03)

single_loss = []

In [ ]:
for epoch in range(200):
    total = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = F.cross_entropy(single_tree(xb), yb)
        loss.backward()
        opt.step()
        total += loss.item() * len(xb)
    single_loss.append(total / len(train_ds))

### Predict and Evaluate

In [ ]:
X_test_t = torch.tensor(X_test_p, dtype=torch.float32).to(device)

with torch.no_grad():
    single_pred = single_tree(X_test_t).argmax(dim=1).cpu().numpy()

In [ ]:
single_acc = accuracy_score(y_test, single_pred)
single_f1 = f1_score(y_test, single_pred, average="weighted")

print("Accuracy:", round(single_acc, 4))
print("Weighted F1:", round(single_f1, 4))

In [ ]:
print(classification_report(y_test, single_pred, target_names=iris_classes))

The single Neural Tree gives the first check that the differentiable tree model is learning useful class boundaries.

# 5. Joint and Greedy Neural Boosting on Iris

### Train Joint Neural Boosting

In [ ]:
joint_trees = nn.ModuleList([NeuralTree(4, 3, depth=2) for _ in range(5)]).to(device)
joint_opt = torch.optim.Adam(joint_trees.parameters(), lr=0.03)

joint_loss = []

In [ ]:
for epoch in range(200):
    total = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        out = torch.stack([tree(xb) for tree in joint_trees]).mean(dim=0)
        loss = F.cross_entropy(out, yb)
        joint_opt.zero_grad()
        loss.backward()
        joint_opt.step()
        total += loss.item() * len(xb)
    joint_loss.append(total / len(train_ds))

### Evaluate Joint Neural Boosting

In [ ]:
with torch.no_grad():
    out = torch.stack([tree(X_test_t) for tree in joint_trees]).mean(dim=0)
    joint_pred = out.argmax(dim=1).cpu().numpy()

In [ ]:
joint_acc = accuracy_score(y_test, joint_pred)
joint_f1 = f1_score(y_test, joint_pred, average="weighted")

print("Accuracy:", round(joint_acc, 4))
print("Weighted F1:", round(joint_f1, 4))

Joint Neural Boosting trains all trees together, then averages their outputs.

### Train Greedy Neural Boosting

In [ ]:
greedy_trees = []

for i in range(5):
    tree = NeuralTree(4, 3, depth=2).to(device)
    opt = torch.optim.Adam(tree.parameters(), lr=0.03)
    greedy_trees.append((tree, opt))

In [ ]:
for tree, opt in greedy_trees:
    for epoch in range(120):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = F.cross_entropy(tree(xb), yb)
            loss.backward()
            opt.step()

### Evaluate Greedy Neural Boosting

In [ ]:
with torch.no_grad():
    out = torch.stack([tree(X_test_t) for tree, opt in greedy_trees]).mean(dim=0)
    greedy_pred = out.argmax(dim=1).cpu().numpy()

In [ ]:
greedy_acc = accuracy_score(y_test, greedy_pred)
greedy_f1 = f1_score(y_test, greedy_pred, average="weighted")

print("Accuracy:", round(greedy_acc, 4))
print("Weighted F1:", round(greedy_f1, 4))

Greedy Neural Boosting trains several trees separately and combines them by averaging their predictions.

# 6. Baseline Models on Iris

### Logistic Regression

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=SEED)
log_reg.fit(X_train_p, y_train)

log_pred = log_reg.predict(X_test_p)

In [ ]:
log_acc = accuracy_score(y_test, log_pred)
log_f1 = f1_score(y_test, log_pred, average="weighted")

### Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=SEED)
rf.fit(X_train_p, y_train)

rf_pred = rf.predict(X_test_p)

In [ ]:
rf_acc = accuracy_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred, average="weighted")

### Gradient Boosting

In [ ]:
gb = GradientBoostingClassifier(random_state=SEED)
gb.fit(X_train_p, y_train)

gb_pred = gb.predict(X_test_p)

In [ ]:
gb_acc = accuracy_score(y_test, gb_pred)
gb_f1 = f1_score(y_test, gb_pred, average="weighted")

### Iris Comparison Table

In [ ]:
iris_results = [
    ["Single Neural Tree", single_acc, single_f1],
    ["Joint Neural Boosting", joint_acc, joint_f1],
    ["Greedy Neural Boosting", greedy_acc, greedy_f1],
    ["Logistic Regression", log_acc, log_f1],
    ["Random Forest", rf_acc, rf_f1],
    ["Gradient Boosting", gb_acc, gb_f1],
]

In [ ]:
iris_results = pd.DataFrame(iris_results, columns=["model", "accuracy", "f1_weighted"])
iris_results

The Iris table confirms that all Neural Tree variants and baseline models are evaluated on the same split.

# 7. Full Experiments Across Datasets

The next part compares Neural Tree models with baseline models across the project datasets. The code is kept simple and direct so each step is easy to follow.

## 7.1 Simple Preprocessing Utilities

In [ ]:
def prepare_data(X_data, y_data, task):
    if len(X_data) > MAX_ROWS:
        X_data = X_data.sample(MAX_ROWS, random_state=SEED)
        y_data = y_data.loc[X_data.index]
    y_ready = y_data if task == "regression" else y_data.astype("category").cat.codes
    return train_test_split(X_data, y_ready, test_size=0.2, random_state=SEED)

In [ ]:
def make_preprocessor(X_train):
    num_cols = X_train.select_dtypes(include=["number"]).columns
    cat_cols = X_train.select_dtypes(exclude=["number"]).columns
    return ColumnTransformer([("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)])

## 7.2 Preprocessing Pipelines

In [ ]:
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

In [ ]:
try:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse=False)

In [ ]:
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("one_hot", one_hot),
])

## 7.3 Simple Neural Tree Training Utilities

In [ ]:
def fit_tree(X_train, y_train, task, depth=2):
    out_size = 1 if task == "regression" else len(np.unique(y_train))
    model = NeuralTree(X_train.shape[1], out_size, depth=depth).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.03)
    losses = []
    return model, opt, losses

In [ ]:
def make_loader(X_train, y_train, task):
    X_t = torch.tensor(X_train, dtype=torch.float32)
    y_type = torch.float32 if task == "regression" else torch.long
    y_t = torch.tensor(np.array(y_train), dtype=y_type)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=64, shuffle=True)

In [ ]:
def train_model(model, opt, loader, task):
    total = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        raw = model(xb)
        pred = raw.squeeze(1) if task == "regression" else raw
        loss = F.mse_loss(pred, yb) if task == "regression" else F.cross_entropy(pred, yb)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * len(xb)
    return total / len(loader.dataset)

These helpers are deliberately small. They only avoid repeating the same PyTorch setup many times.

## 7.4 Start Result Storage

In [ ]:
results = []
saved_predictions = {}
loss_examples = {}
saved_trees = {}

## 7.5 Iris Experiment

### Select Dataset

In [ ]:
name = "Iris"
task = "classification"
X_data = iris_X
y_data = iris_y

### Sample Larger Data

In [ ]:
if len(X_data) > MAX_ROWS:
    X_data = X_data.sample(MAX_ROWS, random_state=SEED)
    y_data = y_data.loc[X_data.index]

### Encode Target

In [ ]:
y_ready = y_data.astype("category").cat.codes

### Train-Test Split

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X_data, y_ready, test_size=0.2, random_state=SEED
)

### Preprocess Features

In [ ]:
prep = make_preprocessor(X_tr)

X_tr_p = prep.fit_transform(X_tr)
X_te_p = prep.transform(X_te)

### Create PyTorch Loader

In [ ]:
loader2 = make_loader(X_tr_p, y_tr, task)

n_features = X_tr_p.shape[1]
n_outputs = 1 if task == "regression" else len(np.unique(y_tr))

### Single Neural Tree

In [ ]:
tree, opt, _ = fit_tree(X_tr_p, y_tr, task)

for epoch in range(EPOCHS):
    last_loss = train_model(tree, opt, loader2, task)

saved_trees[name] = tree
loss_examples[name] = [last_loss]

### Predict with Single Neural Tree

In [ ]:
X_te_t = torch.tensor(X_te_p, dtype=torch.float32).to(device)

with torch.no_grad():
    pred = tree(X_te_t).squeeze().cpu().numpy()

In [ ]:
pred = pred.reshape(-1, n_outputs).argmax(axis=1)
saved_predictions[(name, "Single Neural Tree")] = (y_te, pred)

acc = accuracy_score(y_te, pred)
f1 = f1_score(y_te, pred, average="weighted")
results += [[name, task, "Single Neural Tree", "accuracy", acc]]
results += [[name, task, "Single Neural Tree", "f1_weighted", f1]]

### Joint Neural Boosting

In [ ]:
joint = nn.ModuleList([
    fit_tree(X_tr_p, y_tr, task)[0] for _ in range(5)
]).to(device)

joint_opt = torch.optim.Adam(joint.parameters(), lr=0.03)

In [ ]:
for epoch in range(EPOCHS):
    for xb, yb in loader2:
        xb, yb = xb.to(device), yb.to(device)
        items = [m(xb).squeeze(1) if task == "regression" else m(xb) for m in joint]
        out = torch.stack(items).mean(dim=0)
        loss = F.mse_loss(out, yb) if task == "regression" else F.cross_entropy(out, yb)
        joint_opt.zero_grad(); loss.backward(); joint_opt.step()

### Predict with Joint Neural Boosting

In [ ]:
with torch.no_grad():
    items = [m(X_te_t).squeeze(1) if task == "regression" else m(X_te_t) for m in joint]
    out = torch.stack(items).mean(dim=0)
    pred = out.cpu().numpy()

In [ ]:
pred = pred.reshape(-1, n_outputs).argmax(axis=1)
saved_predictions[(name, "Joint Neural Boosting")] = (y_te, pred)

results += [[name, task, "Joint Neural Boosting", "accuracy", accuracy_score(y_te, pred)]]
results += [[name, task, "Joint Neural Boosting", "f1_weighted", f1_score(y_te, pred, average="weighted")]]

### Greedy Neural Boosting

In [ ]:
greedy = []

for i in range(5):
    model, opt, _ = fit_tree(X_tr_p, y_tr, task)
    for epoch in range(EPOCHS // 2):
        train_model(model, opt, loader2, task)
    greedy.append(model)

### Predict with Greedy Neural Boosting

In [ ]:
with torch.no_grad():
    items = [m(X_te_t).squeeze(1) if task == "regression" else m(X_te_t) for m in greedy]
    out = torch.stack(items).mean(dim=0)
    pred = out.cpu().numpy()

In [ ]:
pred = pred.reshape(-1, n_outputs).argmax(axis=1)
saved_predictions[(name, "Greedy Neural Boosting")] = (y_te, pred)

results += [[name, task, "Greedy Neural Boosting", "accuracy", accuracy_score(y_te, pred)]]
results += [[name, task, "Greedy Neural Boosting", "f1_weighted", f1_score(y_te, pred, average="weighted")]]

### Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_tr_p, y_tr)
pred = model.predict(X_te_p)

saved_predictions[(name, "Logistic Regression")] = (y_te, pred)
results += [[name, task, "Logistic Regression", "accuracy", accuracy_score(y_te, pred)]]
results += [[name, task, "Logistic Regression", "f1_weighted", f1_score(y_te, pred, average="weighted")]]

### Random Forest

In [ ]:
model = RandomForestClassifier(n_estimators=200, random_state=SEED)
model.fit(X_tr_p, y_tr)
pred = model.predict(X_te_p)

saved_predictions[(name, "Random Forest")] = (y_te, pred)
results += [[name, task, "Random Forest", "accuracy", accuracy_score(y_te, pred)]]
results += [[name, task, "Random Forest", "f1_weighted", f1_score(y_te, pred, average="weighted")]]

### Gradient Boosting

In [ ]:
model = GradientBoostingClassifier(random_state=SEED)
model.fit(X_tr_p, y_tr)
pred = model.predict(X_te_p)

saved_predictions[(name, "Gradient Boosting",)] = (y_te, pred)
results += [[name, task, "Gradient Boosting", "accuracy", accuracy_score(y_te, pred)]]
results += [[name, task, "Gradient Boosting", "f1_weighted", f1_score(y_te, pred, average="weighted")]]

### XGBoost

In [ ]:
if has_xgb:
    model = XGBClassifier(eval_metric="mlogloss", random_state=SEED)
    model.fit(X_tr_p, y_tr)
    pred = model.predict(X_te_p)
    saved_predictions[(name, "XGBoost")] = (y_te, pred)
    results += [[name, task, "XGBoost", "accuracy", accuracy_score(y_te, pred)]]
    results += [[name, task, "XGBoost", "f1_weighted", f1_score(y_te, pred, average="weighted")]]

The Iris scores have been added to the final comparison table.

## 7.6 Wine Quality Experiment

### Select Dataset

In [ ]:
name = "Wine Quality"
task = "regression"
X_data = wine_X
y_data = wine_y

### Sample Larger Data

In [ ]:
if len(X_data) > MAX_ROWS:
    X_data = X_data.sample(MAX_ROWS, random_state=SEED)
    y_data = y_data.loc[X_data.index]

### Keep Numeric Target

In [ ]:
y_ready = y_data

### Train-Test Split

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X_data, y_ready, test_size=0.2, random_state=SEED
)

### Preprocess Features

In [ ]:
prep = make_preprocessor(X_tr)

X_tr_p = prep.fit_transform(X_tr)
X_te_p = prep.transform(X_te)

### Create PyTorch Loader

In [ ]:
loader2 = make_loader(X_tr_p, y_tr, task)

n_features = X_tr_p.shape[1]
n_outputs = 1 if task == "regression" else len(np.unique(y_tr))

### Single Neural Tree

In [ ]:
tree, opt, _ = fit_tree(X_tr_p, y_tr, task)

for epoch in range(EPOCHS):
    last_loss = train_model(tree, opt, loader2, task)

saved_trees[name] = tree
loss_examples[name] = [last_loss]

### Predict with Single Neural Tree

In [ ]:
X_te_t = torch.tensor(X_te_p, dtype=torch.float32).to(device)

with torch.no_grad():
    pred = tree(X_te_t).squeeze().cpu().numpy()

In [ ]:
saved_predictions[(name, "Single Neural Tree")] = (y_te, pred)

mse = mean_squared_error(y_te, pred)
r2 = r2_score(y_te, pred)
results += [[name, task, "Single Neural Tree", "mse", mse]]
results += [[name, task, "Single Neural Tree", "r2", r2]]

### Joint Neural Boosting

In [ ]:
joint = nn.ModuleList([
    fit_tree(X_tr_p, y_tr, task)[0] for _ in range(5)
]).to(device)

joint_opt = torch.optim.Adam(joint.parameters(), lr=0.03)

In [ ]:
for epoch in range(EPOCHS):
    for xb, yb in loader2:
        xb, yb = xb.to(device), yb.to(device)
        items = [m(xb).squeeze(1) if task == "regression" else m(xb) for m in joint]
        out = torch.stack(items).mean(dim=0)
        loss = F.mse_loss(out, yb) if task == "regression" else F.cross_entropy(out, yb)
        joint_opt.zero_grad(); loss.backward(); joint_opt.step()

### Predict with Joint Neural Boosting

In [ ]:
with torch.no_grad():
    items = [m(X_te_t).squeeze(1) if task == "regression" else m(X_te_t) for m in joint]
    out = torch.stack(items).mean(dim=0)
    pred = out.cpu().numpy()

In [ ]:
saved_predictions[(name, "Joint Neural Boosting")] = (y_te, pred)

results += [[name, task, "Joint Neural Boosting", "mse", mean_squared_error(y_te, pred)]]
results += [[name, task, "Joint Neural Boosting", "r2", r2_score(y_te, pred)]]

### Greedy Neural Boosting

In [ ]:
greedy = []

for i in range(5):
    model, opt, _ = fit_tree(X_tr_p, y_tr, task)
    for epoch in range(EPOCHS // 2):
        train_model(model, opt, loader2, task)
    greedy.append(model)

### Predict with Greedy Neural Boosting

In [ ]:
with torch.no_grad():
    items = [m(X_te_t).squeeze(1) if task == "regression" else m(X_te_t) for m in greedy]
    out = torch.stack(items).mean(dim=0)
    pred = out.cpu().numpy()

In [ ]:
saved_predictions[(name, "Greedy Neural Boosting")] = (y_te, pred)

results += [[name, task, "Greedy Neural Boosting", "mse", mean_squared_error(y_te, pred)]]
results += [[name, task, "Greedy Neural Boosting", "r2", r2_score(y_te, pred)]]

### Linear Regression

In [ ]:
model = LinearRegression()
model.fit(X_tr_p, y_tr)
pred = model.predict(X_te_p)

saved_predictions[(name, "Linear Regression")] = (y_te, pred)
results += [[name, task, "Linear Regression", "mse", mean_squared_error(y_te, pred)]]
results += [[name, task, "Linear Regression", "r2", r2_score(y_te, pred)]]

### Random Forest

In [ ]:
model = RandomForestRegressor(n_estimators=200, random_state=SEED)
model.fit(X_tr_p, y_tr)
pred = model.predict(X_te_p)

saved_predictions[(name, "Random Forest")] = (y_te, pred)
results += [[name, task, "Random Forest", "mse", mean_squared_error(y_te, pred)]]
results += [[name, task, "Random Forest", "r2", r2_score(y_te, pred)]]

### Gradient Boosting

In [ ]:
model = GradientBoostingRegressor(random_state=SEED)
model.fit(X_tr_p, y_tr)
pred = model.predict(X_te_p)

saved_predictions[(name, "Gradient Boosting")] = (y_te, pred)
results += [[name, task, "Gradient Boosting", "mse", mean_squared_error(y_te, pred)]]
results += [[name, task, "Gradient Boosting", "r2", r2_score(y_te, pred)]]

### XGBoost

In [ ]:
if has_xgb:
    model = XGBRegressor(random_state=SEED)
    model.fit(X_tr_p, y_tr)
    pred = model.predict(X_te_p)
    saved_predictions[(name, "XGBoost")] = (y_te, pred)
    results += [[name, task, "XGBoost", "mse", mean_squared_error(y_te, pred)]]
    results += [[name, task, "XGBoost", "r2", r2_score(y_te, pred)]]

The Wine Quality scores have been added to the final comparison table.

## 7.7 Student Performance Experiment

### Select Dataset

In [ ]:
name = "Student Performance"
task = "regression"
X_data = student_X
y_data = student_y

### Sample Larger Data

In [ ]:
if len(X_data) > MAX_ROWS:
    X_data = X_data.sample(MAX_ROWS, random_state=SEED)
    y_data = y_data.loc[X_data.index]

### Keep Numeric Target

In [ ]:
y_ready = y_data

### Train-Test Split

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X_data, y_ready, test_size=0.2, random_state=SEED
)

### Preprocess Features

In [ ]:
prep = make_preprocessor(X_tr)

X_tr_p = prep.fit_transform(X_tr)
X_te_p = prep.transform(X_te)

### Create PyTorch Loader

In [ ]:
loader2 = make_loader(X_tr_p, y_tr, task)

n_features = X_tr_p.shape[1]
n_outputs = 1 if task == "regression" else len(np.unique(y_tr))

### Single Neural Tree

In [ ]:
tree, opt, _ = fit_tree(X_tr_p, y_tr, task)

for epoch in range(EPOCHS):
    last_loss = train_model(tree, opt, loader2, task)

saved_trees[name] = tree
loss_examples[name] = [last_loss]

### Predict with Single Neural Tree

In [ ]:
X_te_t = torch.tensor(X_te_p, dtype=torch.float32).to(device)

with torch.no_grad():
    pred = tree(X_te_t).squeeze().cpu().numpy()

In [ ]:
saved_predictions[(name, "Single Neural Tree")] = (y_te, pred)

mse = mean_squared_error(y_te, pred)
r2 = r2_score(y_te, pred)
results += [[name, task, "Single Neural Tree", "mse", mse]]
results += [[name, task, "Single Neural Tree", "r2", r2]]

### Joint Neural Boosting

In [ ]:
joint = nn.ModuleList([
    fit_tree(X_tr_p, y_tr, task)[0] for _ in range(5)
]).to(device)

joint_opt = torch.optim.Adam(joint.parameters(), lr=0.03)

In [ ]:
for epoch in range(EPOCHS):
    for xb, yb in loader2:
        xb, yb = xb.to(device), yb.to(device)
        items = [m(xb).squeeze(1) if task == "regression" else m(xb) for m in joint]
        out = torch.stack(items).mean(dim=0)
        loss = F.mse_loss(out, yb) if task == "regression" else F.cross_entropy(out, yb)
        joint_opt.zero_grad(); loss.backward(); joint_opt.step()

### Predict with Joint Neural Boosting

In [ ]:
with torch.no_grad():
    items = [m(X_te_t).squeeze(1) if task == "regression" else m(X_te_t) for m in joint]
    out = torch.stack(items).mean(dim=0)
    pred = out.cpu().numpy()

In [ ]:
saved_predictions[(name, "Joint Neural Boosting")] = (y_te, pred)

results += [[name, task, "Joint Neural Boosting", "mse", mean_squared_error(y_te, pred)]]
results += [[name, task, "Joint Neural Boosting", "r2", r2_score(y_te, pred)]]

### Greedy Neural Boosting

In [ ]:
greedy = []

for i in range(5):
    model, opt, _ = fit_tree(X_tr_p, y_tr, task)
    for epoch in range(EPOCHS // 2):
        train_model(model, opt, loader2, task)
    greedy.append(model)

### Predict with Greedy Neural Boosting

In [ ]:
with torch.no_grad():
    items = [m(X_te_t).squeeze(1) if task == "regression" else m(X_te_t) for m in greedy]
    out = torch.stack(items).mean(dim=0)
    pred = out.cpu().numpy()

In [ ]:
saved_predictions[(name, "Greedy Neural Boosting")] = (y_te, pred)

results += [[name, task, "Greedy Neural Boosting", "mse", mean_squared_error(y_te, pred)]]
results += [[name, task, "Greedy Neural Boosting", "r2", r2_score(y_te, pred)]]

### Linear Regression

In [ ]:
model = LinearRegression()
model.fit(X_tr_p, y_tr)
pred = model.predict(X_te_p)

saved_predictions[(name, "Linear Regression")] = (y_te, pred)
results += [[name, task, "Linear Regression", "mse", mean_squared_error(y_te, pred)]]
results += [[name, task, "Linear Regression", "r2", r2_score(y_te, pred)]]

### Random Forest

In [ ]:
model = RandomForestRegressor(n_estimators=200, random_state=SEED)
model.fit(X_tr_p, y_tr)
pred = model.predict(X_te_p)

saved_predictions[(name, "Random Forest")] = (y_te, pred)
results += [[name, task, "Random Forest", "mse", mean_squared_error(y_te, pred)]]
results += [[name, task, "Random Forest", "r2", r2_score(y_te, pred)]]

### Gradient Boosting

In [ ]:
model = GradientBoostingRegressor(random_state=SEED)
model.fit(X_tr_p, y_tr)
pred = model.predict(X_te_p)

saved_predictions[(name, "Gradient Boosting")] = (y_te, pred)
results += [[name, task, "Gradient Boosting", "mse", mean_squared_error(y_te, pred)]]
results += [[name, task, "Gradient Boosting", "r2", r2_score(y_te, pred)]]

### XGBoost

In [ ]:
if has_xgb:
    model = XGBRegressor(random_state=SEED)
    model.fit(X_tr_p, y_tr)
    pred = model.predict(X_te_p)
    saved_predictions[(name, "XGBoost")] = (y_te, pred)
    results += [[name, task, "XGBoost", "mse", mean_squared_error(y_te, pred)]]
    results += [[name, task, "XGBoost", "r2", r2_score(y_te, pred)]]

The Student Performance scores have been added to the final comparison table.

## 7.8 Adult Experiment

### Select Dataset

In [ ]:
name = "Adult"
task = "classification"
X_data = adult_X
y_data = adult_y

### Sample Larger Data

In [ ]:
if len(X_data) > MAX_ROWS:
    X_data = X_data.sample(MAX_ROWS, random_state=SEED)
    y_data = y_data.loc[X_data.index]

### Encode Target

In [ ]:
y_ready = y_data.astype("category").cat.codes

### Train-Test Split

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X_data, y_ready, test_size=0.2, random_state=SEED
)

### Preprocess Features

In [ ]:
prep = make_preprocessor(X_tr)

X_tr_p = prep.fit_transform(X_tr)
X_te_p = prep.transform(X_te)

### Create PyTorch Loader

In [ ]:
loader2 = make_loader(X_tr_p, y_tr, task)

n_features = X_tr_p.shape[1]
n_outputs = 1 if task == "regression" else len(np.unique(y_tr))

### Single Neural Tree

In [ ]:
tree, opt, _ = fit_tree(X_tr_p, y_tr, task)

for epoch in range(EPOCHS):
    last_loss = train_model(tree, opt, loader2, task)

saved_trees[name] = tree
loss_examples[name] = [last_loss]

### Predict with Single Neural Tree

In [ ]:
X_te_t = torch.tensor(X_te_p, dtype=torch.float32).to(device)

with torch.no_grad():
    pred = tree(X_te_t).squeeze().cpu().numpy()

In [ ]:
pred = pred.reshape(-1, n_outputs).argmax(axis=1)
saved_predictions[(name, "Single Neural Tree")] = (y_te, pred)

acc = accuracy_score(y_te, pred)
f1 = f1_score(y_te, pred, average="weighted")
results += [[name, task, "Single Neural Tree", "accuracy", acc]]
results += [[name, task, "Single Neural Tree", "f1_weighted", f1]]

### Joint Neural Boosting

In [ ]:
joint = nn.ModuleList([
    fit_tree(X_tr_p, y_tr, task)[0] for _ in range(5)
]).to(device)

joint_opt = torch.optim.Adam(joint.parameters(), lr=0.03)

In [ ]:
for epoch in range(EPOCHS):
    for xb, yb in loader2:
        xb, yb = xb.to(device), yb.to(device)
        items = [m(xb).squeeze(1) if task == "regression" else m(xb) for m in joint]
        out = torch.stack(items).mean(dim=0)
        loss = F.mse_loss(out, yb) if task == "regression" else F.cross_entropy(out, yb)
        joint_opt.zero_grad(); loss.backward(); joint_opt.step()

### Predict with Joint Neural Boosting

In [ ]:
with torch.no_grad():
    items = [m(X_te_t).squeeze(1) if task == "regression" else m(X_te_t) for m in joint]
    out = torch.stack(items).mean(dim=0)
    pred = out.cpu().numpy()

In [ ]:
pred = pred.reshape(-1, n_outputs).argmax(axis=1)
saved_predictions[(name, "Joint Neural Boosting")] = (y_te, pred)

results += [[name, task, "Joint Neural Boosting", "accuracy", accuracy_score(y_te, pred)]]
results += [[name, task, "Joint Neural Boosting", "f1_weighted", f1_score(y_te, pred, average="weighted")]]

### Greedy Neural Boosting

In [ ]:
greedy = []

for i in range(5):
    model, opt, _ = fit_tree(X_tr_p, y_tr, task)
    for epoch in range(EPOCHS // 2):
        train_model(model, opt, loader2, task)
    greedy.append(model)

### Predict with Greedy Neural Boosting

In [ ]:
with torch.no_grad():
    items = [m(X_te_t).squeeze(1) if task == "regression" else m(X_te_t) for m in greedy]
    out = torch.stack(items).mean(dim=0)
    pred = out.cpu().numpy()

In [ ]:
pred = pred.reshape(-1, n_outputs).argmax(axis=1)
saved_predictions[(name, "Greedy Neural Boosting")] = (y_te, pred)

results += [[name, task, "Greedy Neural Boosting", "accuracy", accuracy_score(y_te, pred)]]
results += [[name, task, "Greedy Neural Boosting", "f1_weighted", f1_score(y_te, pred, average="weighted")]]

### Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_tr_p, y_tr)
pred = model.predict(X_te_p)

saved_predictions[(name, "Logistic Regression")] = (y_te, pred)
results += [[name, task, "Logistic Regression", "accuracy", accuracy_score(y_te, pred)]]
results += [[name, task, "Logistic Regression", "f1_weighted", f1_score(y_te, pred, average="weighted")]]

### Random Forest

In [ ]:
model = RandomForestClassifier(n_estimators=200, random_state=SEED)
model.fit(X_tr_p, y_tr)
pred = model.predict(X_te_p)

saved_predictions[(name, "Random Forest")] = (y_te, pred)
results += [[name, task, "Random Forest", "accuracy", accuracy_score(y_te, pred)]]
results += [[name, task, "Random Forest", "f1_weighted", f1_score(y_te, pred, average="weighted")]]

### Gradient Boosting

In [ ]:
model = GradientBoostingClassifier(random_state=SEED)
model.fit(X_tr_p, y_tr)
pred = model.predict(X_te_p)

saved_predictions[(name, "Gradient Boosting",)] = (y_te, pred)
results += [[name, task, "Gradient Boosting", "accuracy", accuracy_score(y_te, pred)]]
results += [[name, task, "Gradient Boosting", "f1_weighted", f1_score(y_te, pred, average="weighted")]]

### XGBoost

In [ ]:
if has_xgb:
    model = XGBClassifier(eval_metric="mlogloss", random_state=SEED)
    model.fit(X_tr_p, y_tr)
    pred = model.predict(X_te_p)
    saved_predictions[(name, "XGBoost")] = (y_te, pred)
    results += [[name, task, "XGBoost", "accuracy", accuracy_score(y_te, pred)]]
    results += [[name, task, "XGBoost", "f1_weighted", f1_score(y_te, pred, average="weighted")]]

The Adult scores have been added to the final comparison table.

## 7.9 Bank Marketing Experiment

### Select Dataset

In [ ]:
name = "Bank Marketing"
task = "classification"
X_data = bank_X
y_data = bank_y

### Sample Larger Data

In [ ]:
if len(X_data) > MAX_ROWS:
    X_data = X_data.sample(MAX_ROWS, random_state=SEED)
    y_data = y_data.loc[X_data.index]

### Encode Target

In [ ]:
y_ready = y_data.astype("category").cat.codes

### Train-Test Split

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X_data, y_ready, test_size=0.2, random_state=SEED
)

### Preprocess Features

In [ ]:
prep = make_preprocessor(X_tr)

X_tr_p = prep.fit_transform(X_tr)
X_te_p = prep.transform(X_te)

### Create PyTorch Loader

In [ ]:
loader2 = make_loader(X_tr_p, y_tr, task)

n_features = X_tr_p.shape[1]
n_outputs = 1 if task == "regression" else len(np.unique(y_tr))

### Single Neural Tree

In [ ]:
tree, opt, _ = fit_tree(X_tr_p, y_tr, task)

for epoch in range(EPOCHS):
    last_loss = train_model(tree, opt, loader2, task)

saved_trees[name] = tree
loss_examples[name] = [last_loss]

### Predict with Single Neural Tree

In [ ]:
X_te_t = torch.tensor(X_te_p, dtype=torch.float32).to(device)

with torch.no_grad():
    pred = tree(X_te_t).squeeze().cpu().numpy()

In [ ]:
pred = pred.reshape(-1, n_outputs).argmax(axis=1)
saved_predictions[(name, "Single Neural Tree")] = (y_te, pred)

acc = accuracy_score(y_te, pred)
f1 = f1_score(y_te, pred, average="weighted")
results += [[name, task, "Single Neural Tree", "accuracy", acc]]
results += [[name, task, "Single Neural Tree", "f1_weighted", f1]]

### Joint Neural Boosting

In [ ]:
joint = nn.ModuleList([
    fit_tree(X_tr_p, y_tr, task)[0] for _ in range(5)
]).to(device)

joint_opt = torch.optim.Adam(joint.parameters(), lr=0.03)

In [ ]:
for epoch in range(EPOCHS):
    for xb, yb in loader2:
        xb, yb = xb.to(device), yb.to(device)
        items = [m(xb).squeeze(1) if task == "regression" else m(xb) for m in joint]
        out = torch.stack(items).mean(dim=0)
        loss = F.mse_loss(out, yb) if task == "regression" else F.cross_entropy(out, yb)
        joint_opt.zero_grad(); loss.backward(); joint_opt.step()

### Predict with Joint Neural Boosting

In [ ]:
with torch.no_grad():
    items = [m(X_te_t).squeeze(1) if task == "regression" else m(X_te_t) for m in joint]
    out = torch.stack(items).mean(dim=0)
    pred = out.cpu().numpy()

In [ ]:
pred = pred.reshape(-1, n_outputs).argmax(axis=1)
saved_predictions[(name, "Joint Neural Boosting")] = (y_te, pred)

results += [[name, task, "Joint Neural Boosting", "accuracy", accuracy_score(y_te, pred)]]
results += [[name, task, "Joint Neural Boosting", "f1_weighted", f1_score(y_te, pred, average="weighted")]]

### Greedy Neural Boosting

In [ ]:
greedy = []

for i in range(5):
    model, opt, _ = fit_tree(X_tr_p, y_tr, task)
    for epoch in range(EPOCHS // 2):
        train_model(model, opt, loader2, task)
    greedy.append(model)

### Predict with Greedy Neural Boosting

In [ ]:
with torch.no_grad():
    items = [m(X_te_t).squeeze(1) if task == "regression" else m(X_te_t) for m in greedy]
    out = torch.stack(items).mean(dim=0)
    pred = out.cpu().numpy()

In [ ]:
pred = pred.reshape(-1, n_outputs).argmax(axis=1)
saved_predictions[(name, "Greedy Neural Boosting")] = (y_te, pred)

results += [[name, task, "Greedy Neural Boosting", "accuracy", accuracy_score(y_te, pred)]]
results += [[name, task, "Greedy Neural Boosting", "f1_weighted", f1_score(y_te, pred, average="weighted")]]

### Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_tr_p, y_tr)
pred = model.predict(X_te_p)

saved_predictions[(name, "Logistic Regression")] = (y_te, pred)
results += [[name, task, "Logistic Regression", "accuracy", accuracy_score(y_te, pred)]]
results += [[name, task, "Logistic Regression", "f1_weighted", f1_score(y_te, pred, average="weighted")]]

### Random Forest

In [ ]:
model = RandomForestClassifier(n_estimators=200, random_state=SEED)
model.fit(X_tr_p, y_tr)
pred = model.predict(X_te_p)

saved_predictions[(name, "Random Forest")] = (y_te, pred)
results += [[name, task, "Random Forest", "accuracy", accuracy_score(y_te, pred)]]
results += [[name, task, "Random Forest", "f1_weighted", f1_score(y_te, pred, average="weighted")]]

### Gradient Boosting

In [ ]:
model = GradientBoostingClassifier(random_state=SEED)
model.fit(X_tr_p, y_tr)
pred = model.predict(X_te_p)

saved_predictions[(name, "Gradient Boosting",)] = (y_te, pred)
results += [[name, task, "Gradient Boosting", "accuracy", accuracy_score(y_te, pred)]]
results += [[name, task, "Gradient Boosting", "f1_weighted", f1_score(y_te, pred, average="weighted")]]

### XGBoost

In [ ]:
if has_xgb:
    model = XGBClassifier(eval_metric="mlogloss", random_state=SEED)
    model.fit(X_tr_p, y_tr)
    pred = model.predict(X_te_p)
    saved_predictions[(name, "XGBoost")] = (y_te, pred)
    results += [[name, task, "XGBoost", "accuracy", accuracy_score(y_te, pred)]]
    results += [[name, task, "XGBoost", "f1_weighted", f1_score(y_te, pred, average="weighted")]]

The Bank Marketing scores have been added to the final comparison table.

## 7.10 Breast Cancer WDBC Experiment

### Select Dataset

In [ ]:
name = "Breast Cancer WDBC"
task = "classification"
X_data = cancer_X
y_data = cancer_y

### Sample Larger Data

In [ ]:
if len(X_data) > MAX_ROWS:
    X_data = X_data.sample(MAX_ROWS, random_state=SEED)
    y_data = y_data.loc[X_data.index]

### Encode Target

In [ ]:
y_ready = y_data.astype("category").cat.codes

### Train-Test Split

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X_data, y_ready, test_size=0.2, random_state=SEED
)

### Preprocess Features

In [ ]:
prep = make_preprocessor(X_tr)

X_tr_p = prep.fit_transform(X_tr)
X_te_p = prep.transform(X_te)

### Create PyTorch Loader

In [ ]:
loader2 = make_loader(X_tr_p, y_tr, task)

n_features = X_tr_p.shape[1]
n_outputs = 1 if task == "regression" else len(np.unique(y_tr))

### Single Neural Tree

In [ ]:
tree, opt, _ = fit_tree(X_tr_p, y_tr, task)

for epoch in range(EPOCHS):
    last_loss = train_model(tree, opt, loader2, task)

saved_trees[name] = tree
loss_examples[name] = [last_loss]

### Predict with Single Neural Tree

In [ ]:
X_te_t = torch.tensor(X_te_p, dtype=torch.float32).to(device)

with torch.no_grad():
    pred = tree(X_te_t).squeeze().cpu().numpy()

In [ ]:
pred = pred.reshape(-1, n_outputs).argmax(axis=1)
saved_predictions[(name, "Single Neural Tree")] = (y_te, pred)

acc = accuracy_score(y_te, pred)
f1 = f1_score(y_te, pred, average="weighted")
results += [[name, task, "Single Neural Tree", "accuracy", acc]]
results += [[name, task, "Single Neural Tree", "f1_weighted", f1]]

### Joint Neural Boosting

In [ ]:
joint = nn.ModuleList([
    fit_tree(X_tr_p, y_tr, task)[0] for _ in range(5)
]).to(device)

joint_opt = torch.optim.Adam(joint.parameters(), lr=0.03)

In [ ]:
for epoch in range(EPOCHS):
    for xb, yb in loader2:
        xb, yb = xb.to(device), yb.to(device)
        items = [m(xb).squeeze(1) if task == "regression" else m(xb) for m in joint]
        out = torch.stack(items).mean(dim=0)
        loss = F.mse_loss(out, yb) if task == "regression" else F.cross_entropy(out, yb)
        joint_opt.zero_grad(); loss.backward(); joint_opt.step()

### Predict with Joint Neural Boosting

In [ ]:
with torch.no_grad():
    items = [m(X_te_t).squeeze(1) if task == "regression" else m(X_te_t) for m in joint]
    out = torch.stack(items).mean(dim=0)
    pred = out.cpu().numpy()

In [ ]:
pred = pred.reshape(-1, n_outputs).argmax(axis=1)
saved_predictions[(name, "Joint Neural Boosting")] = (y_te, pred)

results += [[name, task, "Joint Neural Boosting", "accuracy", accuracy_score(y_te, pred)]]
results += [[name, task, "Joint Neural Boosting", "f1_weighted", f1_score(y_te, pred, average="weighted")]]

### Greedy Neural Boosting

In [ ]:
greedy = []

for i in range(5):
    model, opt, _ = fit_tree(X_tr_p, y_tr, task)
    for epoch in range(EPOCHS // 2):
        train_model(model, opt, loader2, task)
    greedy.append(model)

### Predict with Greedy Neural Boosting

In [ ]:
with torch.no_grad():
    items = [m(X_te_t).squeeze(1) if task == "regression" else m(X_te_t) for m in greedy]
    out = torch.stack(items).mean(dim=0)
    pred = out.cpu().numpy()

In [ ]:
pred = pred.reshape(-1, n_outputs).argmax(axis=1)
saved_predictions[(name, "Greedy Neural Boosting")] = (y_te, pred)

results += [[name, task, "Greedy Neural Boosting", "accuracy", accuracy_score(y_te, pred)]]
results += [[name, task, "Greedy Neural Boosting", "f1_weighted", f1_score(y_te, pred, average="weighted")]]

### Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_tr_p, y_tr)
pred = model.predict(X_te_p)

saved_predictions[(name, "Logistic Regression")] = (y_te, pred)
results += [[name, task, "Logistic Regression", "accuracy", accuracy_score(y_te, pred)]]
results += [[name, task, "Logistic Regression", "f1_weighted", f1_score(y_te, pred, average="weighted")]]

### Random Forest

In [ ]:
model = RandomForestClassifier(n_estimators=200, random_state=SEED)
model.fit(X_tr_p, y_tr)
pred = model.predict(X_te_p)

saved_predictions[(name, "Random Forest")] = (y_te, pred)
results += [[name, task, "Random Forest", "accuracy", accuracy_score(y_te, pred)]]
results += [[name, task, "Random Forest", "f1_weighted", f1_score(y_te, pred, average="weighted")]]

### Gradient Boosting

In [ ]:
model = GradientBoostingClassifier(random_state=SEED)
model.fit(X_tr_p, y_tr)
pred = model.predict(X_te_p)

saved_predictions[(name, "Gradient Boosting",)] = (y_te, pred)
results += [[name, task, "Gradient Boosting", "accuracy", accuracy_score(y_te, pred)]]
results += [[name, task, "Gradient Boosting", "f1_weighted", f1_score(y_te, pred, average="weighted")]]

### XGBoost

In [ ]:
if has_xgb:
    model = XGBClassifier(eval_metric="mlogloss", random_state=SEED)
    model.fit(X_tr_p, y_tr)
    pred = model.predict(X_te_p)
    saved_predictions[(name, "XGBoost")] = (y_te, pred)
    results += [[name, task, "XGBoost", "accuracy", accuracy_score(y_te, pred)]]
    results += [[name, task, "XGBoost", "f1_weighted", f1_score(y_te, pred, average="weighted")]]

The Breast Cancer WDBC scores have been added to the final comparison table.

All listed models are trained for every dataset where they apply. The result table uses a long format so classification and regression metrics do not create unnecessary missing values.

### Complete Results Table

In [ ]:
results_df = pd.DataFrame(results, columns=["dataset", "task", "model", "metric", "value"])
results_df

This complete table is the main record of the experiments. It shows that every dataset has results for the Neural Tree methods and the matching baseline models, so the comparison is not missing models because of empty or skipped runs.

### Classification Accuracy

In [ ]:
accuracy_table = results_df[results_df["metric"] == "accuracy"]
accuracy_table.pivot(index="dataset", columns="model", values="value")

For classification, the Neural Tree models are competitive on the smaller structured datasets. On Iris and Breast Cancer, the Neural Tree and boosting variants perform very strongly. On Adult and Bank Marketing, the baseline ensembles, especially Gradient Boosting or XGBoost, are usually stronger or at least as strong.

### Classification Weighted F1

In [ ]:
f1_table = results_df[results_df["metric"] == "f1_weighted"]
f1_table.pivot(index="dataset", columns="model", values="value")

Weighted F1 gives a better view when classes are imbalanced. The Adult and Bank datasets show why this matters: a model can have good accuracy while still being weaker on the smaller class. The boosting Neural Tree versions sometimes improve over the Single Neural Tree, but the improvement is not consistent across every dataset.

### Regression R2

In [ ]:
r2_table = results_df[results_df["metric"] == "r2"]
r2_table.pivot(index="dataset", columns="model", values="value")

The regression results are the clearest weakness of the Neural Tree approach in this notebook. On Wine Quality, the Single and Joint Neural Tree models have positive R² values, but they are below the stronger baseline models. On Student Performance, the Neural Tree and boosting models have negative R² values, which means they perform worse than simply predicting the average grade.

### Regression MSE

In [ ]:
mse_table = results_df[results_df["metric"] == "mse"]
mse_table.pivot(index="dataset", columns="model", values="value")

The MSE table supports the same conclusion as the R² table. Lower MSE is better, and the regression baselines have much smaller errors on Student Performance. This suggests that the current Neural Tree setup is not flexible or well-tuned enough for that regression task.

# 8. Result Visualizations

### Accuracy by Model

In [ ]:
plt.figure(figsize=(10, 5))

for model in accuracy_table["model"].unique():
    rows = accuracy_table[accuracy_table["model"] == model]
    plt.bar(rows["dataset"] + "\n" + model[:12], rows["value"])

plt.xticks(rotation=90)
plt.ylabel("Accuracy")
plt.title("Classification Accuracy")
plt.tight_layout()
plt.show()

The accuracy plot makes the classification pattern easier to see. Neural Trees are strong on Iris and Breast Cancer, while the baseline models remain very reliable on Adult and Bank Marketing. Greedy Neural Boosting helps on Bank Marketing, but boosting does not automatically improve every dataset.

### Regression R2 by Model

In [ ]:
plt.figure(figsize=(8, 4))

for model in r2_table["model"].unique():
    rows = r2_table[r2_table["model"] == model]
    plt.plot(rows["dataset"], rows["value"], marker="o", label=model)

plt.ylabel("R2")
plt.title("Regression Performance")
plt.legend()
plt.tight_layout()
plt.show()

The regression plot highlights the negative R² problem clearly. A negative R² means the model is doing worse than predicting the mean target value for every test case. This happens for the Neural Tree models on Student Performance, while Linear Regression, Random Forest, Gradient Boosting, and XGBoost explain the data much better.

# 9. Explainability

### Neural Tree Split Weights

In [ ]:
tree = saved_trees["Iris"]
weights = tree.splits.weight.detach().cpu().numpy()

pd.DataFrame(weights, columns=iris_X.columns)

The split weights show which Iris measurements each soft decision node uses most strongly. This gives the Neural Tree some interpretability because we can inspect how the model is making its soft splits instead of treating it as a completely hidden black box.

### Split Weight Heatmap

In [ ]:
plt.figure(figsize=(6, 4))
plt.imshow(weights, cmap="coolwarm", aspect="auto")
plt.colorbar(label="weight")
plt.xticks(range(len(iris_X.columns)), iris_X.columns, rotation=30)
plt.yticks(range(weights.shape[0]), [f"split {i}" for i in range(weights.shape[0])])
plt.title("Iris Neural Tree Split Weights")
plt.tight_layout()
plt.show()

Red and blue cells represent positive and negative split weights. Larger absolute values mean stronger influence.

### Leaf Probabilities for Test Examples

In [ ]:
with torch.no_grad():
    p = torch.sigmoid(tree.splits(X_test_t)).cpu()

leaf_0 = (1 - p[:, 0]) * (1 - p[:, 1])
leaf_1 = (1 - p[:, 0]) * p[:, 1]
leaf_2 = p[:, 0] * (1 - p[:, 2])
leaf_3 = p[:, 0] * p[:, 2]
leaf_probs = torch.stack([leaf_0, leaf_1, leaf_2, leaf_3], dim=1).numpy()

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(range(4), leaf_probs.mean(axis=0))
plt.xlabel("Leaf")
plt.ylabel("Average probability")
plt.title("Average Iris Leaf Probabilities")
plt.show()

The leaf probabilities show how much the test examples use each leaf on average. If one leaf dominates completely, the tree is behaving more like a simple model; if several leaves are used, the tree is taking advantage of its structure.

### Random Forest Feature Importance

In [ ]:
rf_importance = pd.Series(rf.feature_importances_, index=iris_X.columns)
rf_importance = rf_importance.sort_values(ascending=True)

rf_importance.plot(kind="barh", figsize=(6, 4), title="Iris Random Forest Importance")
plt.show()

This baseline importance plot gives another view of which Iris features are most useful.

# 10. Training Diagnostics

### Neural Tree Loss Curves

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(single_loss, label="Single Neural Tree")
plt.plot(joint_loss, label="Joint Neural Boosting")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Iris Training Loss")
plt.legend()
plt.show()

The loss curves are a training diagnostic. When the curves decrease, it means the soft splits and leaf predictions are improving during optimization. If a curve stayed flat or unstable, that would suggest the learning rate, depth, or model setup needs attention.

### Confusion Matrices for Adult and Bank

In [ ]:
pairs = [
    ("Adult", "Single Neural Tree"),
    ("Adult", "Logistic Regression"),
    ("Adult", "Random Forest"),
    ("Bank Marketing", "Single Neural Tree"),
    ("Bank Marketing", "Logistic Regression"),
    ("Bank Marketing", "Random Forest"),
]

fig, axes = plt.subplots(2, 3, figsize=(10, 5.5))

for ax, (dataset_name, model_name) in zip(axes.ravel(), pairs):
    y_true, y_pred = saved_predictions[(dataset_name, model_name)]
    cm = confusion_matrix(y_true, y_pred)
    ax.imshow(cm, cmap="Blues")
    ax.set_title(dataset_name + "\n" + model_name, fontsize=9)
    ax.set_xlabel("Predicted", fontsize=8)
    ax.set_ylabel("True", fontsize=8)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.tick_params(labelsize=8)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=9)

plt.tight_layout(pad=2.0)
plt.show()

These confusion matrices focus on Adult and Bank Marketing because both are harder and more imbalanced than Iris. A clean model should not only predict the majority class; it should also identify some of the smaller positive class. Comparing Neural Trees with Logistic Regression and Random Forest helps show where the Neural Tree approach is weaker or stronger.

# 11. Depth Ablation Study

The depth ablation checks how the Neural Tree changes when we allow more leaves. This uses Iris so the comparison is quick and easy to inspect.

In [ ]:
depth_rows = []
depth_loader = make_loader(X_train_p, y_train, "classification")

for depth in [1, 2, 3]:
    model = NeuralTree(4, 3, depth=depth).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.03)
    for epoch in range(120):
        train_model(model, opt, depth_loader, "classification")
    with torch.no_grad():
        pred = model(X_test_t).argmax(dim=1).cpu().numpy()
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, average="weighted")
    depth_rows.append([depth, 2**depth, acc, f1])

In [ ]:
depth_df = pd.DataFrame(depth_rows, columns=["depth", "leaves", "accuracy", "f1_weighted"])
depth_df

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(depth_df["depth"], depth_df["accuracy"], marker="o")
plt.xlabel("Tree depth")
plt.ylabel("Accuracy")
plt.title("Depth Ablation on Iris")
plt.show()

The depth ablation shows that increasing the tree depth from 1 to 3 did not improve Iris accuracy in this run. This suggests that a shallow Neural Tree is already enough for Iris, and extra leaves add complexity without a clear benefit on this small dataset.

# 12. Final Interpretation and Conclusion

The experiments show that Neural Trees can work well on classification problems, especially on Iris and Breast Cancer WDBC. On these datasets, the Single Neural Tree and the boosting versions reached strong accuracy and weighted F1 scores, sometimes matching or beating the baseline models.

The results were more mixed on Adult and Bank Marketing. Joint and Greedy Neural Boosting sometimes improved over the Single Neural Tree, but the improvement was not guaranteed. The classical baselines, especially Random Forest, Gradient Boosting, Logistic Regression, and XGBoost, remained very competitive and often gave the strongest or most stable classification results.

The regression datasets were the main weakness of the Neural Tree models. On Wine Quality, the Neural Tree models achieved some positive R² values, but the baseline regression models performed better. On Student Performance, the Neural Tree and boosting models had negative R² values, meaning they were worse than predicting the mean grade; this shows that the current Neural Tree setup did not capture that regression problem well.

The depth ablation on Iris showed that increasing the depth from 1 to 3 did not improve performance. This suggests that more leaves do not automatically make a Neural Tree better, especially on a small dataset where a shallow tree is already enough.

Overall, Neural Trees are useful because they combine neural-network training with a tree-like structure that can still be inspected through split weights and leaf probabilities. Their weakness is that they can be sensitive to training settings and may not perform as strongly as established baselines, especially for regression. For this project, the Neural Tree approach is interesting and interpretable, but Random Forest, Gradient Boosting, and XGBoost are generally more reliable across all datasets.